# shard-build-sft — một shard của build_sft.py (kaggle_orchestra)

Bản shard của `build_data.ipynb`: MỘT tập (`params.nguon`) mỗi job, khoảng `[tu, den)`
từ cell ORCHESTRA-PARAMS. Prompt hybrid (BM25 + e5 + RRF + rerank) khớp `infer_hybrid`.

In [ ]:
# ORCHESTRA-PARAMS — kaggle_orchestra.py thay cell này lúc đẩy. Giá trị dưới chỉ để chạy tay.
SHARD = {"job": "tay", "chi_so": 0, "so_shard": 1, "tu": 0, "den": 4, "commit": "", "noi_tiep": None, "output": "x.jsonl"}

In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path

# Codebase chép ra /tmp chứ KHÔNG ra /kaggle/working: mọi thứ trong working là output, và
# orchestrator tải output của từng shard về máy dev sau mỗi lần dừng — 30 MB repo × số lần.
SRC = next((c.parent.parent.parent for c in Path("/kaggle/input").glob("**/legalqa/bm25.py")
            if "legalqa-v10-src" in str(c)), None)
assert SRC, "Không thấy codebase — gắn dataset legalqa-v10-src"
REPO = Path("/tmp/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
commit = (REPO / "COMMIT.txt").read_text().strip()
print("codebase:", commit, "| shard", SHARD)
# Mọi shard phải chạy CÙNG commit: dataset src của từng acc được đẩy riêng, lệch một acc
# là ghép hai phiên bản code vào một tập mà không dấu vết.
assert not SHARD["commit"] or commit == SHARD["commit"], f"src {commit} != {SHARD['commit']}"

OUT = Path("/kaggle/working/out"); OUT.mkdir(parents=True, exist_ok=True)
# Nối tiếp: dataset `orch-<job>-s<i>` mang phần đã làm từ acc trước.
if SHARD["noi_tiep"]:
    for f in Path("/kaggle/input").glob("**/*.jsonl"):
        if SHARD["noi_tiep"] in str(f):
            shutil.copy(f, OUT / f.name)
            print(f"nối tiếp {f.name}: {sum(1 for _ in open(OUT / f.name, encoding='utf-8')):,} dòng")

In [ ]:
!pip -q install transformers==4.44.2 sentencepiece

In [ ]:
from build_sft import MAX_CHARS, TOP_N
assert TOP_N == 5 and MAX_CHARS > 10**6, f"codebase CŨ: top_n={TOP_N} max_chars={MAX_CHARS}"

ch = next((p for p in Path("/kaggle/input").glob("**/chunks.jsonl")), None)
assert ch, "Không thấy chunks.jsonl — gắn dataset legalqa-clean"
# Symlink ra /tmp chứ không ra working: working là output, symlink trong đó bị Kaggle
# đóng gói lại thành bản sao GB.
CLEAN = Path("/tmp/clean"); CLEAN.mkdir(parents=True, exist_ok=True)
for f in ch.parent.iterdir():
    if not (CLEAN / f.name).exists():
        (CLEAN / f.name).symlink_to(f)
os.environ["CLEAN_DIR"] = str(CLEAN)
os.environ["PYTHONPATH"] = "src"

local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text()]
if local:
    MODEL = str(local[0])
else:
    from huggingface_hub import snapshot_download
    MODEL = snapshot_download("AITeamVN/Vietnamese_Reranker")
e5 = [p.parent for p in Path("/kaggle/input").glob("**/config.json") if "multilingual-e5-small" in str(p)]
if e5:
    DENSE = str(e5[0])
else:
    from huggingface_hub import snapshot_download
    DENSE = snapshot_download("intfloat/multilingual-e5-small")
tk = next((p for p in Path("/kaggle/input").glob("**/tokenizer.json") if "qwen" in str(p).lower()), None)
assert tk, "Không thấy tokenizer Qwen — gắn model qwen3-5-2b-base"
os.environ["QWEN_TOKENIZER"] = str(tk)
print("rerank", MODEL, "| dense", DENSE, "| tok", tk)

In [ ]:
# subprocess.run(check=True): `!python` không ném theo mã thoát, kernel xanh mà file rỗng.
lenh = [sys.executable, "scripts/build_sft.py",
    "--model", MODEL, "--dense", DENSE, "--batch-size", "32", "--pool", "50",
    "--nguon", SHARD["nguon"], "--gold", SHARD["gold"],
    "--tu", str(SHARD["tu"]), "--den", str(SHARD["den"]), "--noi-tiep",
    "--ra", str(OUT / SHARD["output"])]
if SHARD.get("giu_het"):
    lenh.append("--giu-het")
subprocess.run(lenh, check=True)